#### Import libaries used

In [0]:
from typing import List, Tuple, Optional, Dict

from pyspark.sql import DataFrame
from pyspark.sql import functions as SQL_FUNCTIONS

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import gc

#### Setup the properties/widgets to be used

In [0]:
dbutils.widgets.text("silver_table", "workspace.bda_taxi.taxi_silver")
dbutils.widgets.text("predictions_table", "workspace.bda_taxi.taxi_predictions")
dbutils.widgets.text("label_col", "tipped")

# Train/test split props
dbutils.widgets.text("train_ratio", "0.8")
dbutils.widgets.text("seed", "42")

# Limit rows for quick iteration (set to 0 to disable)
dbutils.widgets.text("limit_rows", "0")

silver_table = dbutils.widgets.get("silver_table").strip()
predictions_table = dbutils.widgets.get("predictions_table").strip()
label_col = dbutils.widgets.get("label_col").strip()

train_ratio = float(dbutils.widgets.get("train_ratio"))
seed = int(dbutils.widgets.get("seed"))
limit_rows = int(dbutils.widgets.get("limit_rows"))

dbutils.widgets.dropdown(
    "model_to_run",
    "logistic_regression",
    ["logistic_regression", "random_forest"]
)
model_to_run = dbutils.widgets.get("model_to_run")
print("Model to run:", model_to_run)

print("Silver table:", silver_table)
print("Predictions table:", predictions_table)
print("Label column:", label_col)
print("Train ratio:", train_ratio, "Seed:", seed, "Limit rows:", limit_rows)

#### Helper functions

In [0]:
def read_table(table_fqn: str) -> DataFrame:
    return spark.table(table_fqn)

def prepare_modelling_dataframe(silver_df: DataFrame, label_column: str, row_limit: int = 0) -> DataFrame:
    """
    Prepares data for modelling:
    - Restrict to tip_recorded == 1 (credit card only)
    - Ensure label exists and is non-null
    - Optionally limit rows for faster iteration
    """
    df = (
        silver_df
        .filter(SQL_FUNCTIONS.col("tip_recorded") == 1)
        .filter(SQL_FUNCTIONS.col(label_column).isNotNull())
    )

    if row_limit and row_limit > 0:
        df = df.limit(row_limit)

    return df

def available_columns(input_dataframe: DataFrame) -> List[str]:
    return input_dataframe.columns

def pick_existing_columns(input_dataframe: DataFrame, candidates: List[str]) -> List[str]:
    cols_set = set(input_dataframe.columns)
    return [c for c in candidates if c in cols_set]

    



#### Helper functions: pipeline creation and evaluation

In [0]:
## Helper functions: pipeline creation and evaluation

# COMMAND ----------
def build_preprocessing_stages(
    numeric_cols: List[str],
    categorical_cols: List[str]
) -> Tuple[List, str]:
    """
    Builds preprocessing stages:
    - StringIndex + OneHot for categoricals
    - VectorAssembler for features
    - StandardScaler (useful for Logistic Regression)
    Returns:
    - stages list
    - name of assembled feature column
    """
    stages = []
    indexed_cols = []
    encoded_cols = []

    for col_name in categorical_cols:
        index_col = f"{col_name}_idx"
        ohe_col = f"{col_name}_ohe"

        # handleInvalid keeps pipeline robust to unseen categories in test data
        indexer = StringIndexer(inputCol=col_name, outputCol=index_col, handleInvalid="keep")
        encoder = OneHotEncoder(inputCols=[index_col], outputCols=[ohe_col], handleInvalid="keep")

        stages.append(indexer)
        stages.append(encoder)
        indexed_cols.append(index_col)
        encoded_cols.append(ohe_col)

    assembled_input_cols = list(numeric_cols) + encoded_cols
    features_raw_col = "features_raw"

    assembler = VectorAssembler(
        inputCols=assembled_input_cols,
        outputCol=features_raw_col,
        handleInvalid="keep"
    )
    stages.append(assembler)

    return stages, features_raw_col


def build_logistic_regression_pipeline(
    numeric_cols: List[str],
    categorical_cols: List[str],
    label_column: str
) -> Pipeline:
    """
    Logistic Regression pipeline:
    - preprocessing
    - scaling
    - LR classifier
    """
    stages, features_raw_col = build_preprocessing_stages(numeric_cols, categorical_cols)

    scaler = StandardScaler(
        inputCol=features_raw_col,
        outputCol="features",
        withMean=False,
        withStd=True
    )
    stages.append(scaler)

    lr = LogisticRegression(
        featuresCol="features",
        labelCol=label_column,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxIter=50,
        regParam=0.05,
        elasticNetParam=0.0
    )
    stages.append(lr)

    return Pipeline(stages=stages)


def build_random_forest_pipeline(
    numeric_cols: List[str],
    categorical_cols: List[str],
    label_column: str
) -> Pipeline:
    """
    Random Forest pipeline:
    - preprocessing
    - RF classifier
    """
    stages, features_raw_col = build_preprocessing_stages(numeric_cols, categorical_cols)

    # RF can work on unscaled features; use raw assembled vector
    rf = RandomForestClassifier(
        featuresCol=features_raw_col,
        labelCol=label_column,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=100,
        maxDepth=10,
        seed=seed
    )
    stages.append(rf)

    return Pipeline(stages=stages)


def evaluate_binary_classifier(predictions_df: DataFrame, label_column: str) -> Dict[str, float]:
    """
    Evaluates a binary classifier with:
    - AUC (ROC)
    - Accuracy
    - F1
    """
    auc_evaluator = BinaryClassificationEvaluator(
        labelCol=label_column,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol=label_column,
        predictionCol="prediction",
        metricName="accuracy"
    )

    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol=label_column,
        predictionCol="prediction",
        metricName="f1"
    )

    return {
        "auc_roc": float(auc_evaluator.evaluate(predictions_df)),
        "accuracy": float(accuracy_evaluator.evaluate(predictions_df)),
        "f1": float(f1_evaluator.evaluate(predictions_df))
    }


def show_confusion_matrix(predictions_df: DataFrame, label_column: str) -> None:
    """
    Displays a simple confusion matrix counts table.
    """
    cm = (
        predictions_df
        .groupBy(
            SQL_FUNCTIONS.col(label_column).alias("label"),
            SQL_FUNCTIONS.col("prediction").cast("int").alias("prediction")
        )
        .count()
        .orderBy("label", "prediction")
    )
    display(cm)

#### Load, prepare modelling dataset and select features

In [0]:
# Load and prepare modelling dataset
silver_dataframe = read_table(silver_table)

model_dataframe = prepare_modelling_dataframe(silver_dataframe, label_col, row_limit=limit_rows)

print("Model rows (tip_recorded only):", model_dataframe.count())
display(model_dataframe.select(label_col, "payment_type", "tip_recorded", "tip_amount").limit(10))

# Feature selection

# Numeric features: stable and commonly used
numeric_feature_candidates = [
    "trip_distance",
    "fare_amount",
    "total_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "passenger_count",
]

# Categorical features: use if present in your Silver table
categorical_feature_candidates = [
    "time_bucket",
    "pickup_hour",
    "pickup_dow",
    "PU_Borough",
    "DO_Borough",
    "PU_Zone",
    "DO_Zone",
    "store_and_fwd_flag",
    "RatecodeID",
    "payment_type"
]

numeric_features = pick_existing_columns(model_dataframe, numeric_feature_candidates)
categorical_features = pick_existing_columns(model_dataframe, categorical_feature_candidates)

# Remove columns that shouldn't be treated as categorical if they are numeric
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

#### Train/test split

In [0]:
train_df, test_df = model_dataframe.randomSplit([train_ratio, 1 - train_ratio], seed=seed)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

####  Baseline model: Logistic Regression

In [0]:

def cleanup_connect_ml_objects(*objects_to_delete) -> None:
    """
    More aggressive cleanup for Spark Connect ML cache issues.
    """
    for obj in objects_to_delete:
        try:
            del obj
        except Exception:
            pass

    try:
        spark.catalog.clearCache()
    except Exception:
        pass

    gc.collect()


def save_predictions_for_comparison(
    predictions_df: DataFrame,
    output_table_fqn: str,
    label_col: str,
    model_name: str
) -> None:
    """
    Save minimal prediction outputs for comparing models later.
    """
    slim_df = (
        predictions_df
        .select(
            SQL_FUNCTIONS.lit(model_name).alias("model_name"),
            SQL_FUNCTIONS.col(label_col).alias("label"),
            SQL_FUNCTIONS.col("prediction").alias("prediction"),
            SQL_FUNCTIONS.col("probability").alias("probability")
        )
    )

    (
        slim_df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(output_table_fqn)
    )


def fit_evaluate_and_cleanup(
    pipeline,
    train_df: DataFrame,
    test_df: DataFrame,
    label_col: str,
    model_name: str,
    predictions_table_fqn: Optional[str] = None
) -> Dict[str, float]:
    """
    Fit + evaluate, optionally persist minimal predictions, then cleanup.
    Returns metrics only (so we don't keep big objects alive).
    """
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    metrics = evaluate_binary_classifier(predictions, label_col)
    print(model_name, "metrics:", metrics)
    show_confusion_matrix(predictions, label_col)

    if predictions_table_fqn:
        save_predictions_for_comparison(
            predictions_df=predictions,
            output_table_fqn=predictions_table_fqn,
            label_col=label_col,
            model_name=model_name
        )

    cleanup_connect_ml_objects(predictions, model, pipeline)
    return metrics

In [0]:
# LR_PRED_TABLE = "workspace.bda_taxi.model_preds_logreg"

# lr_pipeline = build_logistic_regression_pipeline(
#     numeric_cols=numeric_features,
#     categorical_cols=categorical_features,
#     label_column=label_col
# )

# lr_metrics = fit_evaluate_and_cleanup(
#     pipeline=lr_pipeline,
#     train_df=train_df,
#     test_df=test_df,
#     label_col=label_col,
#     model_name="LogisticRegression",
#     predictions_table_fqn=LR_PRED_TABLE
# )

### Run when need to delete previous model due to Databricks Free Tier cacheing limit

#### Improved model: Random Forest

In [0]:
# RF_PRED_TABLE = "workspace.bda_taxi.model_preds_rf"

# rf_pipeline = build_random_forest_pipeline(
#     numeric_cols=numeric_features,
#     categorical_cols=categorical_features,
#     label_column=label_col
# )

# rf_metrics = fit_evaluate_and_cleanup(
#     pipeline=rf_pipeline,
#     train_df=train_df,
#     test_df=test_df,
#     label_col=label_col,
#     model_name="RandomForest",
#     predictions_table_fqn=RF_PRED_TABLE
# )


#### Compare Models

In [0]:
# comparison_df = spark.createDataFrame([
#     ("LogisticRegression", float(lr_metrics["auc_roc"]), float(lr_metrics["accuracy"]), float(lr_metrics["f1"])),
#     ("RandomForest",       float(rf_metrics["auc_roc"]), float(rf_metrics["accuracy"]), float(rf_metrics["f1"]))
# ], ["model", "auc_roc", "accuracy", "f1"])

# display(comparison_df.orderBy(SQL_FUNCTIONS.desc("auc_roc")))

#### Testing out model selection per run

In [0]:

def ensure_schema_exists_for_table(table_fqn: str) -> None:
    """
    Ensures the schema exists for a fully qualified table name: catalog.schema.table
    Example: workspace.bda_taxi.model_metrics_comparison
    """
    parts = table_fqn.split(".")
    if len(parts) < 3:
        raise ValueError(
            f"Expected fully qualified table name (catalog.schema.table), got: {table_fqn}"
        )

    schema_fqn = ".".join(parts[:2])  # catalog.schema
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_fqn}")


def write_delta_table_overwrite(input_dataframe: DataFrame, target_table_fqn: str) -> None:
    """
    Overwrites a Delta table (creates schema if needed).
    """
    ensure_schema_exists_for_table(target_table_fqn)
    (
        input_dataframe.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(target_table_fqn)
    )


def cleanup_connect_ml_objects(*objects_to_delete) -> None:
    """
    Cleanup for Spark Connect ML cache issues.

    On Serverless, spark.catalog.clearCache() is not supported.
    We just delete references + force Python GC.
    """
    for obj in objects_to_delete:
        try:
            del obj
        except Exception:
            pass

    gc.collect()


def upsert_model_metrics(metrics_table_fqn: str, model_name: str, metrics: Dict[str, float]) -> None:
    """
    Insert-or-update a single row of metrics for one model in a Delta table.

    Why overwrite the whole table?
    - The table is tiny (2 rows), so this is simplest and reliable on Serverless.
    """
    ensure_schema_exists_for_table(metrics_table_fqn)

    # Create table if missing
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {metrics_table_fqn} (
          model STRING,
          auc_roc DOUBLE,
          accuracy DOUBLE,
          f1 DOUBLE
        )
        USING DELTA
    """)

    # New row for this model
    row_df = spark.createDataFrame(
        [(model_name, float(metrics["auc_roc"]), float(metrics["accuracy"]), float(metrics["f1"]))],
        ["model", "auc_roc", "accuracy", "f1"]
    )

    # Remove any existing row for this model, then union the new row
    existing_df = spark.table(metrics_table_fqn).filter(SQL_FUNCTIONS.col("model") != model_name)
    merged_df = existing_df.unionByName(row_df)

    write_delta_table_overwrite(merged_df, metrics_table_fqn)

In [0]:
METRICS_TABLE = "workspace.bda_taxi.model_metrics_comparison"

LOGREG_PRED_TABLE = "workspace.bda_taxi.model_preds_logreg"
RF_PRED_TABLE = "workspace.bda_taxi.model_preds_rf"

if model_to_run == "logistic_regression":
    model_name = "LogisticRegression"
    pred_table = LOGREG_PRED_TABLE

    pipeline = build_logistic_regression_pipeline(
        numeric_cols=numeric_features,
        categorical_cols=categorical_features,
        label_column=label_col
    )

    metrics = fit_evaluate_and_cleanup(
        pipeline=pipeline,
        train_df=train_df,
        test_df=test_df,
        label_col=label_col,
        model_name=model_name,
        predictions_table_fqn=pred_table
    )

    upsert_model_metrics(METRICS_TABLE, model_name, metrics)

elif model_to_run == "random_forest":
    model_name = "RandomForest"
    pred_table = RF_PRED_TABLE

    pipeline = build_random_forest_pipeline(
        numeric_cols=numeric_features,
        categorical_cols=categorical_features,
        label_column=label_col
    )

    metrics = fit_evaluate_and_cleanup(
        pipeline=pipeline,
        train_df=train_df,
        test_df=test_df,
        label_col=label_col,
        model_name=model_name,
        predictions_table_fqn=pred_table
    )

    upsert_model_metrics(METRICS_TABLE, model_name, metrics)

else:
    raise ValueError(f"Unknown model_to_run: {model_to_run}")

print("Saved metrics to:", METRICS_TABLE)
print("Saved predictions to:", pred_table)

### Run after both models run


In [0]:
# metrics_comparison = spark.table(METRICS_TABLE)
# display(metrics_comparison.orderBy(SQL_FUNCTIONS.desc("auc_roc")))

In [0]:
# preds_all = spark.table(LOGREG_PRED_TABLE).unionByName(spark.table(RF_PRED_TABLE))
# display(
#     preds_all.groupBy("model_name")
#     .agg(
#         SQL_FUNCTIONS.count("*").alias("rows"),
#         SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label").cast("double")).alias("actual_tip_rate"),
#         SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("prediction").cast("double")).alias("predicted_tip_rate")
#     )
#     .orderBy("model_name")
# )